# Effective two-level pMOT trajectory laboratory

This notebook is the interactive trajectory front end for `src/pmot/pMOT_2Level`. Choose an initial 3D position and velocity, optical parameters, propagation-frame helicities, duration, and timestep; then press **Run trajectory**.

Each run produces:

- a speed-colored 3D trajectory inside translucent 780-nm cooling and 1529-nm trapping-beam envelopes;
- an animation with the atom's location, trail, elapsed time, coordinates, and speed;
- a force-versus-time graph overlaid with the x/y/z velocities;
- a Cartesian history figure with x/y/z positions overlaid in one panel and $F_x/F_y/F_z$ overlaid in a second panel;
- an optional compressed trajectory file, figures, embedded animation HTML, and metadata.

**Scientific boundary:** this is the provisional effective $5S_{1/2},F=2\rightarrow5P_{3/2},F'=3$ model. It has no external magnetic field and uses the differential scalar/vector/tensor Stark closure, but it is not an $m_F$-resolved Hamiltonian. It omits the conservative 1529-nm gradient force, trapping-light scattering/heating, recoil diffusion, coherent standing-wave structure, measured polarization transformations, and nonadiabatic dynamics at the optical-spin zero. The default cooling power is now $1.0571$ mW per traveling beam ($s_0\simeq1$ for the 12.7-mm Gaussian beam). The default trapping-path powers remain $(P_x,P_y,P_z)=(19.147243,19.147243,38.294486)$ mW, chosen to produce the provisional effective-gradient target $(+10,+10,-20)$ G/cm. These are diagnostic settings, not apparatus recommendations.

In [ ]:
import json
import os
import sys
from dataclasses import replace
from datetime import datetime
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent:
    if (PROJECT_ROOT / 'pyproject.toml').is_file() and (PROJECT_ROOT / 'src' / 'pmot').is_dir():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
else:
    raise RuntimeError('Could not locate the pMOT repository root.')

SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

try:
    backend = 'inline' if os.environ.get('PMOT_NOTEBOOK_HEADLESS') == '1' else 'widget'
    get_ipython().run_line_magic('matplotlib', backend)
except (ImportError, ModuleNotFoundError):
    backend = 'inline'
    get_ipython().run_line_magic('matplotlib', 'inline')

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, Markdown, display

from pmot.configuration import RB87_MASS_KG
from pmot.mot_simple.configuration import default_simple_mot_apparatus
from pmot.mot_simple.configuration import default_simple_mot_config
from pmot.mot_strange.trapping_beams import TrappingLaserConfig
from pmot.pMOT_2Level import (
    animate_trajectory,
    build_pmot_2level_cooling_beams,
    build_pmot_2level_trapping_beams,
    load_pmot_2level_polarizability_table,
    plot_force_velocity_history,
    plot_trajectory_3d,
    simulate_pmot_2level_trajectory,
    trajectory_arrays,
)
from pmot.state import AtomState

OUTPUT_ROOT = PROJECT_ROOT / 'outputs' / 'trajectories' / 'pMOT_2Level' / 'notebook'
print(f'Project root: {PROJECT_ROOT}')
print(f'Matplotlib backend: {backend}')
print(f'Run outputs: {OUTPUT_ROOT}')

## What is evaluated at every RK4 stage

The six cooling-beam intensities, Doppler shifts, local optical-spin quantization axis, Doppler-shifted 1529-nm wavelengths, interpolated differential polarizabilities, AC Stark transition shifts, polarization decomposition, scattering rates, radiation force, and gravity are all recomputed at every RK4 stage. The ordinary-frequency channel detuning is

$$\Delta_{\mathrm{eff},b,q}=\Delta_{0,b}-\frac{\mathbf{k}_b\cdot\mathbf{v}}{2\pi}-\frac{\Delta E_{\mathrm{AC},q}}{h}.$$

Rates from all six cooling components and all local $q=-1,0,+1$ fractions are summed deterministically. This is a mean-force trajectory; it does not choose individual scattered photons.

In [ ]:
CORE_RADIUS_M = 2.0e-3
CORE_RESIDENCE_S = 5.0e-3
DEFAULT_XYZ_HELICITIES = ('sigma+', 'sigma+', 'sigma-')
DEFAULT_COOLING_POWER_MW_PER_BEAM = 1.0571


def _core_summary(record):
    arrays = trajectory_arrays(record)
    time_s = arrays['time_s']
    inside = np.linalg.norm(arrays['position_m'], axis=1) <= CORE_RADIUS_M
    entries = int(inside[0]) + int(np.sum((~inside[:-1]) & inside[1:]))
    longest = 0.0
    start = None
    for index, value in enumerate(inside):
        if value and start is None:
            start = index
        if start is not None and (not value or index == len(inside) - 1):
            end = index if value else index - 1
            longest = max(longest, float(time_s[end] - time_s[start]))
            start = None
    return {
        'core entry count': entries,
        'longest continuous core residence [ms]': 1e3 * longest,
        'meets early trapped criterion': bool(longest >= CORE_RESIDENCE_S or entries >= 2),
    }


def _json_ready(value):
    if isinstance(value, dict):
        return {str(key): _json_ready(item) for key, item in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_ready(item) for item in value]
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, (np.floating, np.integer, np.bool_)):
        return value.item()
    return value


def plot_cartesian_position_force_history(
    record, *, gravity_acceleration_m_per_s2=(0.0, 0.0, 0.0)
):
    """Plot Cartesian position and total-force components versus time."""

    arrays = trajectory_arrays(record)
    time_ms = 1.0e3 * arrays['time_s']
    position_mm = 1.0e3 * arrays['position_m']
    gravity_acceleration = np.asarray(gravity_acceleration_m_per_s2, dtype=float)
    if gravity_acceleration.shape != (3,):
        raise ValueError('gravity_acceleration_m_per_s2 must be a 3-vector')
    total_force_z_n = 1.0e21 * (
        arrays['force_n'] + RB87_MASS_KG * gravity_acceleration
    )
    labels = ('x', 'y', 'z')
    colors = ('tab:red', 'tab:green', 'tab:blue')
    figure, axes = plt.subplots(2, 1, figsize=(11.5, 8.0), sharex=True)
    for component, (label, color) in enumerate(zip(labels, colors)):
        axes[0].plot(
            time_ms, position_mm[:, component], color=color, linewidth=1.5,
            label=label,
        )
        axes[1].plot(
            time_ms, total_force_z_n[:, component], color=color, linewidth=1.5,
            label=f'F_{label}',
        )
    axes[0].set_ylabel('position [mm]')
    axes[0].set_title('Cartesian position history')
    axes[1].set_xlabel('time [ms]')
    axes[1].set_ylabel('total force [zN]')
    axes[1].set_title('Cartesian total-force history')
    for axis in axes:
        axis.axhline(0.0, color='black', linewidth=0.7, alpha=0.55)
        axis.grid(alpha=0.22)
        axis.legend(ncol=3, frameon=False, loc='best')
    figure.tight_layout()
    return figure, axes


def run_pmot_2level_case(
    *,
    initial_position_mm=(15.0, 0.0, 0.0),
    initial_velocity_m_per_s=(-17.0, 0.0, 0.0),
    duration_ms=25.0,
    time_step_us=2.5,
    include_gravity=True,
    cooling_power_mw_per_beam=DEFAULT_COOLING_POWER_MW_PER_BEAM,
    cooling_detuning_mhz=-15.0,
    cooling_beam_diameter_mm=12.7,
    cooling_helicities_xyz=DEFAULT_XYZ_HELICITIES,
    trapping_power_x_mw=19.147243,
    trapping_power_y_mw=19.147243,
    trapping_power_z_mw=38.294486,
    trapping_wavelength_nm=1529.268881,
    trapping_focus_offset_mm=10.0,
    trapping_input_diameter_mm=35.0,
    trapping_focal_length_mm=80.3,
    trapping_retro_power_fraction=1.0,
    trapping_incident_helicities_xyz=DEFAULT_XYZ_HELICITIES,
    trapping_retro_helicities_xyz=DEFAULT_XYZ_HELICITIES,
    plot_extent_mm=20.0,
    maximum_animation_frames=140,
    animation_interval_ms=55,
    save_outputs=True,
    output_tag=None,
):
    if duration_ms <= 0.0 or time_step_us <= 0.0:
        raise ValueError('duration_ms and time_step_us must be positive')
    trapping_path_powers_mw = np.asarray(
        (trapping_power_x_mw, trapping_power_y_mw, trapping_power_z_mw), dtype=float
    )
    if not np.all(np.isfinite(trapping_path_powers_mw)):
        raise ValueError('trapping path powers must be finite')
    if np.any(trapping_path_powers_mw < 0.0) or cooling_power_mw_per_beam < 0.0:
        raise ValueError('beam powers must be non-negative')
    position_m = 1.0e-3 * np.asarray(initial_position_mm, dtype=float)
    velocity_m_per_s = np.asarray(initial_velocity_m_per_s, dtype=float)
    if position_m.shape != (3,) or velocity_m_per_s.shape != (3,):
        raise ValueError('initial position and velocity must each contain x, y, z')

    simple = replace(
        default_simple_mot_config(),
        cooling_detuning_hz=1.0e6 * cooling_detuning_mhz,
        include_gravity=bool(include_gravity),
    )
    baseline_apparatus = default_simple_mot_apparatus()
    apparatus = replace(
        baseline_apparatus,
        cooling=replace(
            baseline_apparatus.cooling,
            power_w_per_beam=1.0e-3 * cooling_power_mw_per_beam,
            detuning_hz=1.0e6 * cooling_detuning_mhz,
            beam_diameter_m=1.0e-3 * cooling_beam_diameter_mm,
        ),
    )
    cooling = build_pmot_2level_cooling_beams(
        apparatus_config=apparatus,
        simple_config=simple,
        helicities_by_axis=cooling_helicities_xyz,
    )
    trapping_config = TrappingLaserConfig(
        wavelength_m=1.0e-9 * trapping_wavelength_nm,
        focus_offset_m=1.0e-3 * trapping_focus_offset_mm,
        input_beam_diameter_m=1.0e-3 * trapping_input_diameter_mm,
        focal_length_m=1.0e-3 * trapping_focal_length_mm,
        retro_power_fraction=trapping_retro_power_fraction,
    )
    trapping = build_pmot_2level_trapping_beams(
        trapping_config,
        incident_helicities_by_axis=trapping_incident_helicities_xyz,
        retro_helicities_by_axis=trapping_retro_helicities_xyz,
    )
    table = load_pmot_2level_polarizability_table()
    path_powers_w = tuple(1.0e-3 * trapping_path_powers_mw)
    effective_gradient_proxy_g_per_cm = (
        20.0 * trapping_path_powers_mw / 38.294486
    )
    effective_gradient_proxy_g_per_cm[2] *= -1.0
    initial_state = AtomState(
        position_m=tuple(position_m),
        velocity_m_per_s=tuple(velocity_m_per_s),
    )

    print(
        f'Integrating {duration_ms:g} ms at dt={time_step_us:g} us '
        f'({int(np.ceil(1e3 * duration_ms / time_step_us)):,} RK4 steps)...'
    )
    record = simulate_pmot_2level_trajectory(
        cooling,
        trapping,
        initial_state,
        1.0e-3 * duration_ms,
        1.0e-6 * time_step_us,
        path_powers_w=path_powers_w,
        simple_config=simple,
        polarizability_table=table,
    )
    arrays = trajectory_arrays(record)
    radius_mm = 1.0e3 * np.linalg.norm(arrays['position_m'], axis=1)
    speed = np.linalg.norm(arrays['velocity_m_per_s'], axis=1)
    summary = {
        'stored samples': len(arrays['time_s']),
        'final time [ms]': 1.0e3 * arrays['time_s'][-1],
        'minimum radius [mm]': float(np.min(radius_mm)),
        'final radius [mm]': float(radius_mm[-1]),
        'initial speed [m/s]': float(speed[0]),
        'final speed [m/s]': float(speed[-1]),
        'maximum speed [m/s]': float(np.max(speed)),
        'peak radiation force [zN]': float(1.0e21 * np.max(np.linalg.norm(arrays['force_n'], axis=1))),
        'gravity enabled': bool(include_gravity),
        'external magnetic field identically zero': True,
        'trapping path powers [mW]': trapping_path_powers_mw.tolist(),
        'calibrated central effective gradient proxy [G/cm]': (
            effective_gradient_proxy_g_per_cm.tolist()
        ),
    }
    summary.update(_core_summary(record))

    trajectory_figure, trajectory_axis = plot_trajectory_3d(
        record, cooling, trapping, extent_mm=plot_extent_mm
    )
    history_figure, history_axes = plot_force_velocity_history(record)
    cartesian_history_figure, cartesian_history_axes = (
        plot_cartesian_position_force_history(
            record,
            gravity_acceleration_m_per_s2=(
                simple.gravity_acceleration_m_per_s2
                if include_gravity else (0.0, 0.0, 0.0)
            ),
        )
    )
    animation = animate_trajectory(
        record,
        cooling,
        trapping,
        extent_mm=plot_extent_mm,
        maximum_frames=int(maximum_animation_frames),
        interval_ms=int(animation_interval_ms),
    )
    animation_html = animation.to_jshtml()
    plt.close(animation._fig)

    output_directory = None
    if save_outputs:
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        tag = output_tag or f'run_{stamp}'
        output_directory = OUTPUT_ROOT / tag
        output_directory.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            output_directory / 'trajectory.npz',
            **arrays,
            quantization_axis=np.asarray(record.quantization_axes, dtype=float),
            axis_defined_by_vector_shift=np.asarray(record.axis_defined_by_vector_shift, dtype=bool),
            sigma_plus_ac_stark_shift_hz=np.asarray(record.total_ac_stark_shifts_hz_for_sigma_plus, dtype=float),
            total_scattering_rate_per_s=np.asarray(record.total_scattering_rates_per_s, dtype=float),
        )
        trajectory_figure.savefig(output_directory / 'trajectory_3d.png', dpi=220, bbox_inches='tight')
        history_figure.savefig(output_directory / 'force_velocity_history.png', dpi=220, bbox_inches='tight')
        cartesian_history_figure.savefig(
            output_directory / 'cartesian_position_force_history.png',
            dpi=220,
            bbox_inches='tight',
        )
        (output_directory / 'trajectory_animation.html').write_text(animation_html, encoding='utf-8')
        metadata = {
            'status': 'PROVISIONAL_EFFECTIVE_TWO_LEVEL_PMOT',
            'summary': summary,
            'initial_position_mm': position_m.tolist(),
            'initial_velocity_m_per_s': velocity_m_per_s.tolist(),
            'duration_ms': duration_ms,
            'time_step_us': time_step_us,
            'cooling_power_mw_per_beam': cooling_power_mw_per_beam,
            'cooling_detuning_mhz': cooling_detuning_mhz,
            'cooling_beam_diameter_mm': cooling_beam_diameter_mm,
            'cooling_helicities_xyz': cooling_helicities_xyz,
            'trapping_path_powers_mw': trapping_path_powers_mw.tolist(),
            'calibrated_central_effective_gradient_proxy_g_per_cm': (
                effective_gradient_proxy_g_per_cm.tolist()
            ),
            'trapping_wavelength_nm': trapping_wavelength_nm,
            'trapping_incident_helicities_xyz': trapping_incident_helicities_xyz,
            'trapping_retro_helicities_xyz': trapping_retro_helicities_xyz,
            'polarizability_csv': str(table.source_path),
        }
        (output_directory / 'metadata.json').write_text(
            json.dumps(_json_ready(metadata), indent=2) + '\n', encoding='utf-8'
        )

    plt.show()
    display(pd.Series(summary, name='value').to_frame())
    display(HTML(animation_html))
    if output_directory is not None:
        print(f'Saved run directory: {output_directory}')
    return {
        'record': record,
        'cooling_beams': cooling,
        'trapping_beams': trapping,
        'summary': summary,
        'trajectory_figure': trajectory_figure,
        'history_figure': history_figure,
        'cartesian_history_figure': cartesian_history_figure,
        'cartesian_history_axes': cartesian_history_axes,
        'animation': animation,
        'animation_html': animation_html,
        'output_directory': output_directory,
    }

## Interactive controls

The position controls are Cartesian coordinates in millimetres and the velocity controls are Cartesian components in metres per second. The two trapping helicity rows specify the propagation-frame polarization of the incident and retroreflected components on the x, y, and z paths. The trapping-path powers are independently adjustable; their defaults target the provisional effective-gradient tensor $(+10,+10,-20)$ G/cm. Nothing runs until you press the button.

In [ ]:
def build_interactive_controls():
    def number(description, value, *, step=0.1, minimum=None, maximum=None):
        kwargs = dict(
            description=description,
            value=value,
            step=step,
            continuous_update=False,
            style={'description_width': '145px'},
            layout=widgets.Layout(width='255px'),
        )
        if minimum is not None and maximum is not None:
            return widgets.BoundedFloatText(min=minimum, max=maximum, **kwargs)
        return widgets.FloatText(**kwargs)

    initial = {
        'x_mm': number('x0 [mm]', 15.0, step=0.5),
        'y_mm': number('y0 [mm]', 0.0, step=0.5),
        'z_mm': number('z0 [mm]', 0.0, step=0.5),
        'vx': number('vx0 [m/s]', -17.0, step=0.25),
        'vy': number('vy0 [m/s]', 0.0, step=0.25),
        'vz': number('vz0 [m/s]', 0.0, step=0.25),
    }
    integration = {
        'duration_ms': number('duration [ms]', 25.0, step=1.0, minimum=0.01, maximum=500.0),
        'time_step_us': number('dt [us]', 2.5, step=0.25, minimum=0.01, maximum=100.0),
        'plot_extent_mm': number('plot extent [mm]', 20.0, step=1.0, minimum=1.0, maximum=200.0),
        'maximum_animation_frames': widgets.BoundedIntText(
            description='animation frames', value=140, min=2, max=400,
            style={'description_width': '145px'}, layout=widgets.Layout(width='255px')
        ),
        'animation_interval_ms': widgets.BoundedIntText(
            description='frame interval [ms]', value=55, min=10, max=1000,
            style={'description_width': '145px'}, layout=widgets.Layout(width='255px')
        ),
        'include_gravity': widgets.Checkbox(description='include gravity', value=True, indent=False),
    }
    optics = {
        'cooling_power_mw_per_beam': number('cooling power [mW]', DEFAULT_COOLING_POWER_MW_PER_BEAM, step=0.0001, minimum=0.0, maximum=1000.0),
        'cooling_detuning_mhz': number('cooling det. [MHz]', -15.0, step=0.5),
        'cooling_beam_diameter_mm': number('cooling diam. [mm]', 12.7, step=0.1, minimum=0.1, maximum=100.0),
        'trapping_power_x_mw': number('x trap power [mW]', 19.147243, step=0.1, minimum=0.0, maximum=10000.0),
        'trapping_power_y_mw': number('y trap power [mW]', 19.147243, step=0.1, minimum=0.0, maximum=10000.0),
        'trapping_power_z_mw': number('z trap power [mW]', 38.294486, step=0.1, minimum=0.0, maximum=10000.0),
        'trapping_wavelength_nm': number('trap wavelength [nm]', 1529.268881, step=0.000001, minimum=1500.0, maximum=1600.0),
        'trapping_focus_offset_mm': number('focus offset [mm]', 10.0, step=0.5, minimum=0.01, maximum=100.0),
        'trapping_input_diameter_mm': number('input diameter [mm]', 35.0, step=0.5, minimum=0.1, maximum=200.0),
        'trapping_focal_length_mm': number('focal length [mm]', 80.3, step=0.1, minimum=0.1, maximum=1000.0),
        'trapping_retro_power_fraction': number('retro power fraction', 1.0, step=0.05, minimum=0.0, maximum=10.0),
    }
    helicity = {}
    helicity_items = [widgets.HTML('<b>family/path</b>'), widgets.HTML('<b>x</b>'), widgets.HTML('<b>y</b>'), widgets.HTML('<b>z</b>')]
    for family_path in ('cooling', 'trapping incident', 'trapping retro'):
        helicity_items.append(widgets.HTML(f'<b>{family_path}</b>'))
        values = DEFAULT_XYZ_HELICITIES
        for axis_index, axis_name in enumerate(('x', 'y', 'z')):
            dropdown = widgets.Dropdown(
                options=('sigma+', 'sigma-', 'pi'),
                value=values[axis_index],
                description='',
                layout=widgets.Layout(width='125px'),
            )
            helicity[(family_path, axis_name)] = dropdown
            helicity_items.append(dropdown)
    helicity_grid = widgets.GridBox(
        helicity_items,
        layout=widgets.Layout(
            grid_template_columns='170px 130px 130px 130px',
            grid_gap='4px 8px',
            align_items='center',
        ),
    )

    preset = widgets.Dropdown(
        description='preset',
        options=('axial 17 m/s', 'oblique launch', 'near local equilibrium', 'custom'),
        value='axial 17 m/s',
        style={'description_width': '145px'},
        layout=widgets.Layout(width='350px'),
    )
    save_outputs = widgets.Checkbox(description='save run outputs', value=True, indent=False)
    run_button = widgets.Button(description='Run trajectory', button_style='primary', icon='play')
    progress = widgets.IntProgress(value=0, min=0, max=100, description='ready')
    output = widgets.Output()

    def apply_preset(change):
        selected = change['new']
        if selected == 'custom':
            return
        if selected == 'axial 17 m/s':
            values = (15.0, 0.0, 0.0, -17.0, 0.0, 0.0)
        elif selected == 'oblique launch':
            direction = np.asarray((2.0, 3.0, 6.0)) / 7.0
            values = tuple(15.0 * direction) + tuple(-12.0 * direction)
        else:
            values = (0.20, 0.0, -0.5664, 0.0, 0.0, 0.0)
            integration['duration_ms'].value = 25.0
        for widget, value in zip(initial.values(), values):
            widget.value = float(value)

    preset.observe(apply_preset, names='value')

    def selected_xyz(row):
        return tuple(helicity[(row, axis)].value for axis in ('x', 'y', 'z'))

    def row(title, controls):
        items = list(controls.values())
        return widgets.VBox(
            [widgets.HTML(f'<b>{title}</b>')] +
            [widgets.HBox(items[index:index + 3]) for index in range(0, len(items), 3)]
        )

    def run_from_widgets(_=None):
        run_button.disabled = True
        progress.value = 5
        progress.description = 'running'
        progress.bar_style = ''
        with output:
            output.clear_output(wait=True)
            plt.close('all')
            try:
                result = run_pmot_2level_case(
                    initial_position_mm=(initial['x_mm'].value, initial['y_mm'].value, initial['z_mm'].value),
                    initial_velocity_m_per_s=(initial['vx'].value, initial['vy'].value, initial['vz'].value),
                    duration_ms=integration['duration_ms'].value,
                    time_step_us=integration['time_step_us'].value,
                    include_gravity=integration['include_gravity'].value,
                    plot_extent_mm=integration['plot_extent_mm'].value,
                    maximum_animation_frames=integration['maximum_animation_frames'].value,
                    animation_interval_ms=integration['animation_interval_ms'].value,
                    **{name: widget.value for name, widget in optics.items()},
                    cooling_helicities_xyz=selected_xyz('cooling'),
                    trapping_incident_helicities_xyz=selected_xyz('trapping incident'),
                    trapping_retro_helicities_xyz=selected_xyz('trapping retro'),
                    save_outputs=save_outputs.value,
                )
                globals()['PMOT_2LEVEL_RUN'] = result
                progress.value = 100
                progress.description = 'done'
                progress.bar_style = 'success'
            except Exception:
                progress.description = 'error'
                progress.bar_style = 'danger'
                raise
            finally:
                run_button.disabled = False

    run_button.on_click(run_from_widgets)
    panel = widgets.VBox([
        widgets.HTML('<b>Preset</b>'), preset,
        row('Initial state', initial),
        row('Integration and display', integration),
        row('Cooling and trapping light', optics),
        widgets.HTML('<b>Propagation-frame helicities</b>'), helicity_grid,
        widgets.HBox([save_outputs, run_button, progress]),
        output,
    ])
    display(panel)
    return {
        'panel': panel, 'preset': preset, 'initial': initial,
        'integration': integration, 'optics': optics, 'helicity': helicity,
    }


PMOT_2LEVEL_CONTROLS = build_interactive_controls()

In [ ]:
display(Markdown(r'''## One-dimensional x-path pMOT diagnostic

This test retains all six 780-nm cooling components but includes only the incident and retroreflected 1529-nm components on the x path. The atom still moves in three dimensions. Disable gravity for the cleanest quantization-axis test. The 3D plot and animation show all eight traveling components; arrows indicate their propagation directions.'''))


def run_x_only_pmot_case(
    *,
    initial_position_mm=(1.0, 0.0, 0.0),
    initial_velocity_m_per_s=(-0.25, 0.0, 0.0),
    duration_ms=10.0,
    time_step_us=2.5,
    include_gravity=False,
    cooling_power_mw_per_beam=DEFAULT_COOLING_POWER_MW_PER_BEAM,
    cooling_detuning_mhz=-15.0,
    cooling_beam_diameter_mm=12.7,
    cooling_helicities_xyz=DEFAULT_XYZ_HELICITIES,
    trapping_power_mw=19.147243,
    trapping_wavelength_nm=1529.268881,
    trapping_focus_offset_mm=10.0,
    trapping_input_diameter_mm=35.0,
    trapping_focal_length_mm=80.3,
    trapping_retro_power_fraction=1.0,
    trapping_incident_helicity='sigma+',
    trapping_retro_helicity='sigma+',
    plot_extent_mm=12.0,
    maximum_animation_frames=120,
    animation_interval_ms=55,
    save_outputs=True,
):
    if duration_ms <= 0.0 or time_step_us <= 0.0:
        raise ValueError('duration_ms and time_step_us must be positive')
    if cooling_power_mw_per_beam < 0.0 or trapping_power_mw < 0.0:
        raise ValueError('beam powers must be non-negative')

    simple = replace(
        default_simple_mot_config(),
        cooling_detuning_hz=1.0e6 * cooling_detuning_mhz,
        include_gravity=bool(include_gravity),
    )
    baseline_apparatus = default_simple_mot_apparatus()
    apparatus = replace(
        baseline_apparatus,
        cooling=replace(
            baseline_apparatus.cooling,
            power_w_per_beam=1.0e-3 * cooling_power_mw_per_beam,
            detuning_hz=1.0e6 * cooling_detuning_mhz,
            beam_diameter_m=1.0e-3 * cooling_beam_diameter_mm,
        ),
    )
    cooling = build_pmot_2level_cooling_beams(
        apparatus_config=apparatus,
        simple_config=simple,
        helicities_by_axis=cooling_helicities_xyz,
    )
    trapping_config = TrappingLaserConfig(
        wavelength_m=1.0e-9 * trapping_wavelength_nm,
        focus_offset_m=1.0e-3 * trapping_focus_offset_mm,
        input_beam_diameter_m=1.0e-3 * trapping_input_diameter_mm,
        focal_length_m=1.0e-3 * trapping_focal_length_mm,
        retro_power_fraction=trapping_retro_power_fraction,
    )
    all_trapping = build_pmot_2level_trapping_beams(
        trapping_config,
        incident_helicities_by_axis=(trapping_incident_helicity, 'pi', 'pi'),
        retro_helicities_by_axis=(trapping_retro_helicity, 'pi', 'pi'),
    )
    trapping = tuple(
        beam for beam in all_trapping if beam.axis_name == 'horizontal_x'
    )
    if len(cooling) != 6 or len(trapping) != 2:
        raise RuntimeError('the x-only diagnostic requires six cooling and two trapping components')

    position_m = 1.0e-3 * np.asarray(initial_position_mm, dtype=float)
    velocity_m_per_s = np.asarray(initial_velocity_m_per_s, dtype=float)
    table = load_pmot_2level_polarizability_table()
    record = simulate_pmot_2level_trajectory(
        cooling,
        trapping,
        AtomState(
            position_m=tuple(position_m),
            velocity_m_per_s=tuple(velocity_m_per_s),
        ),
        1.0e-3 * duration_ms,
        1.0e-6 * time_step_us,
        path_powers_w=(1.0e-3 * trapping_power_mw, 0.0, 0.0),
        simple_config=simple,
        polarizability_table=table,
    )
    arrays = trajectory_arrays(record)
    axes = np.asarray(record.quantization_axes, dtype=float)
    axis_defined = np.asarray(record.axis_defined_by_vector_shift, dtype=bool)
    speed = np.linalg.norm(arrays['velocity_m_per_s'], axis=1)
    summary = {
        'traveling cooling components': len(cooling),
        'traveling trapping components': len(trapping),
        'trapping axes': sorted({beam.axis_name for beam in trapping}),
        'final position [mm]': (1.0e3 * arrays['position_m'][-1]).tolist(),
        'final velocity [m/s]': arrays['velocity_m_per_s'][-1].tolist(),
        'maximum speed [m/s]': float(np.max(speed)),
        'axis-defined sample fraction': float(np.mean(axis_defined)),
        'vector-zero/fallback samples': int(np.count_nonzero(~axis_defined)),
        'maximum |n_yz| when vector-defined': (
            float(np.max(np.linalg.norm(axes[axis_defined, 1:], axis=1)))
            if np.any(axis_defined) else float('nan')
        ),
        'gravity enabled': bool(include_gravity),
    }

    trajectory_figure, trajectory_axis = plot_trajectory_3d(
        record, cooling, trapping, extent_mm=plot_extent_mm
    )
    trajectory_axis.set_title('1D x-path pMOT: six cooling + two trapping components')
    history_figure, history_axes = plot_force_velocity_history(record)
    cartesian_history_figure, cartesian_history_axes = (
        plot_cartesian_position_force_history(
            record,
            gravity_acceleration_m_per_s2=(
                simple.gravity_acceleration_m_per_s2
                if include_gravity else (0.0, 0.0, 0.0)
            ),
        )
    )
    animation = animate_trajectory(
        record,
        cooling,
        trapping,
        extent_mm=plot_extent_mm,
        maximum_frames=int(maximum_animation_frames),
        interval_ms=int(animation_interval_ms),
    )
    animation._fig.axes[0].set_title('1D x-path pMOT: atom location')
    animation_html = animation.to_jshtml()
    plt.close(animation._fig)

    output_directory = None
    if save_outputs:
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        output_directory = OUTPUT_ROOT / f'x_only_{stamp}'
        output_directory.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(
            output_directory / 'trajectory.npz',
            **arrays,
            quantization_axis=axes,
            axis_defined_by_vector_shift=axis_defined,
        )
        trajectory_figure.savefig(
            output_directory / 'trajectory_3d.png', dpi=220, bbox_inches='tight'
        )
        history_figure.savefig(
            output_directory / 'force_velocity_history.png', dpi=220, bbox_inches='tight'
        )
        cartesian_history_figure.savefig(
            output_directory / 'cartesian_position_force_history.png',
            dpi=220,
            bbox_inches='tight',
        )
        (output_directory / 'trajectory_animation.html').write_text(
            animation_html, encoding='utf-8'
        )
        metadata = {
            'status': 'PROVISIONAL_EFFECTIVE_TWO_LEVEL_PMOT_X_ONLY',
            'summary': summary,
            'initial_position_mm': list(initial_position_mm),
            'initial_velocity_m_per_s': list(initial_velocity_m_per_s),
            'duration_ms': duration_ms,
            'time_step_us': time_step_us,
            'cooling_helicities_xyz': list(cooling_helicities_xyz),
            'trapping_incident_helicity_x': trapping_incident_helicity,
            'trapping_retro_helicity_x': trapping_retro_helicity,
            'trapping_power_mw_x_path': trapping_power_mw,
            'polarizability_csv': str(table.source_path),
        }
        (output_directory / 'metadata.json').write_text(
            json.dumps(_json_ready(metadata), indent=2) + '\n', encoding='utf-8'
        )

    plt.show()
    display(pd.Series(summary, name='value').to_frame())
    display(HTML(animation_html))
    if output_directory is not None:
        print(f'Saved x-only run directory: {output_directory}')
    return {
        'record': record,
        'cooling_beams': cooling,
        'trapping_beams': trapping,
        'summary': summary,
        'trajectory_figure': trajectory_figure,
        'history_figure': history_figure,
        'cartesian_history_figure': cartesian_history_figure,
        'cartesian_history_axes': cartesian_history_axes,
        'animation': animation,
        'output_directory': output_directory,
    }


def build_x_only_pmot_controls():
    def number(description, value, *, step=0.1, minimum=None, maximum=None):
        kwargs = dict(
            description=description, value=value, step=step,
            continuous_update=False, style={'description_width': '145px'},
            layout=widgets.Layout(width='255px'),
        )
        if minimum is not None and maximum is not None:
            return widgets.BoundedFloatText(min=minimum, max=maximum, **kwargs)
        return widgets.FloatText(**kwargs)

    controls = {
        'x_mm': number('x0 [mm]', 1.0, step=0.1),
        'y_mm': number('y0 [mm]', 0.0, step=0.1),
        'z_mm': number('z0 [mm]', 0.0, step=0.1),
        'vx': number('vx0 [m/s]', -0.25, step=0.05),
        'vy': number('vy0 [m/s]', 0.0, step=0.05),
        'vz': number('vz0 [m/s]', 0.0, step=0.05),
        'duration_ms': number('duration [ms]', 10.0, step=1.0, minimum=0.01, maximum=500.0),
        'time_step_us': number('dt [us]', 2.5, step=0.25, minimum=0.01, maximum=100.0),
        'plot_extent_mm': number('plot extent [mm]', 12.0, step=1.0, minimum=1.0, maximum=200.0),
        'cooling_power': number('cooling power [mW]', DEFAULT_COOLING_POWER_MW_PER_BEAM, step=0.0001, minimum=0.0, maximum=1000.0),
        'cooling_detuning': number('cooling det. [MHz]', -15.0, step=0.5),
        'trapping_power': number('x trap power [mW]', 19.147243, step=0.1, minimum=0.0, maximum=10000.0),
        'trapping_wavelength': number('trap wavelength [nm]', 1529.268881, step=0.000001, minimum=1500.0, maximum=1600.0),
    }
    cooling_helicity = [
        widgets.Dropdown(options=('sigma+', 'sigma-', 'pi'), value=value, description=axis)
        for axis, value in zip(('x', 'y', 'z'), DEFAULT_XYZ_HELICITIES)
    ]
    trap_incident = widgets.Dropdown(
        options=('sigma+', 'sigma-', 'pi'), value='sigma+', description='x incident'
    )
    trap_retro = widgets.Dropdown(
        options=('sigma+', 'sigma-', 'pi'), value='sigma+', description='x retro'
    )
    include_gravity = widgets.Checkbox(description='include gravity', value=False, indent=False)
    save_outputs = widgets.Checkbox(description='save run outputs', value=True, indent=False)
    run_button = widgets.Button(description='Run x-only trajectory', button_style='primary', icon='play')
    progress = widgets.IntProgress(value=0, min=0, max=100, description='ready')
    output = widgets.Output()

    def rows(names):
        items = [controls[name] for name in names]
        return widgets.VBox([
            widgets.HBox(items[index:index + 3])
            for index in range(0, len(items), 3)
        ])

    def run_from_widgets(_=None):
        run_button.disabled = True
        progress.value = 5
        progress.description = 'running'
        progress.bar_style = ''
        with output:
            output.clear_output(wait=True)
            plt.close('all')
            try:
                result = run_x_only_pmot_case(
                    initial_position_mm=(controls['x_mm'].value, controls['y_mm'].value, controls['z_mm'].value),
                    initial_velocity_m_per_s=(controls['vx'].value, controls['vy'].value, controls['vz'].value),
                    duration_ms=controls['duration_ms'].value,
                    time_step_us=controls['time_step_us'].value,
                    plot_extent_mm=controls['plot_extent_mm'].value,
                    cooling_power_mw_per_beam=controls['cooling_power'].value,
                    cooling_detuning_mhz=controls['cooling_detuning'].value,
                    cooling_helicities_xyz=tuple(widget.value for widget in cooling_helicity),
                    trapping_power_mw=controls['trapping_power'].value,
                    trapping_wavelength_nm=controls['trapping_wavelength'].value,
                    trapping_incident_helicity=trap_incident.value,
                    trapping_retro_helicity=trap_retro.value,
                    include_gravity=include_gravity.value,
                    save_outputs=save_outputs.value,
                )
                globals()['PMOT_1D_X_RUN'] = result
                progress.value = 100
                progress.description = 'done'
                progress.bar_style = 'success'
            except Exception:
                progress.description = 'error'
                progress.bar_style = 'danger'
                raise
            finally:
                run_button.disabled = False

    run_button.on_click(run_from_widgets)
    panel = widgets.VBox([
        widgets.HTML('<b>Initial 3D state</b>'),
        rows(('x_mm', 'y_mm', 'z_mm', 'vx', 'vy', 'vz')),
        widgets.HTML('<b>Integration and light</b>'),
        rows(('duration_ms', 'time_step_us', 'plot_extent_mm', 'cooling_power', 'cooling_detuning', 'trapping_power', 'trapping_wavelength')),
        widgets.HTML('<b>Propagation-frame helicities</b>'),
        widgets.HBox(cooling_helicity),
        widgets.HBox((trap_incident, trap_retro)),
        widgets.HBox((include_gravity, save_outputs, run_button, progress)),
        output,
    ])
    display(panel)
    return {'panel': panel, 'controls': controls, 'output': output}


PMOT_1D_X_CONTROLS = build_x_only_pmot_controls()


## Programmatic use

The widget panel is the normal entry point. The following call is intentionally commented out so **Run All** does not launch a potentially expensive trajectory. Uncomment and edit it if you prefer a script-like workflow.

In [ ]:
# PMOT_2LEVEL_RUN = run_pmot_2level_case(
#     initial_position_mm=(15.0, 0.0, 0.0),
#     initial_velocity_m_per_s=(-17.0, 0.0, 0.0),
#     duration_ms=25.0,
#     time_step_us=2.5,
#     cooling_power_mw_per_beam=DEFAULT_COOLING_POWER_MW_PER_BEAM,
#     cooling_detuning_mhz=-15.0,
#     trapping_power_x_mw=19.147243,
#     trapping_power_y_mw=19.147243,
#     trapping_power_z_mw=38.294486,
#     save_outputs=True,
# )

## Reading the result

The translucent blue cylinders are the three counterpropagating cooling paths; the orange and purple surfaces are the true-scale incident and retroreflected 1529-nm Gaussian $1/e^2$ envelopes. The 35-mm trapping diameter is the pre-lens input and is therefore **not** drawn through the trap.

The force curves are the modeled 780-nm radiation forces and exclude gravity; gravity is included separately in the trajectory acceleration when selected. Solid lines are $F_x,F_y,F_z$ in zeptonewtons, while matching dashed lines are $v_x,v_y,v_z$ in metres per second.

The animation is downsampled only for display. The simulation and saved `trajectory.npz` retain every RK4 endpoint. A displayed `meets early trapped criterion = True` means 5 ms continuous residence inside the central 2-mm core or two entries with an intervening exit; it is a diagnostic classification, not proof of a physical pMOT.